# RETECO · Diver-Retriever-4B

Encodes all 507K track 2 documents with `AQ-MedAI/Diver-Retriever-4B` and retrieves the
top 100 for every turn. Everything is saved to Google Drive under `reteco/`.

**How to run.** Runtime → Change runtime type → **T4 GPU**. Then
Runtime → **Run all**. Allow Drive access, then leave it running.

**If the session drops**, choose "Run all" again. Finished domains are on Drive
and are skipped; the run resumes where it stopped. Nothing is computed twice.

**How long.** On a T4, an estimated 2–4 hours for the whole corpus; the notebook
prints the exact speed after the first domain. An A100 is 5–8× faster. About
2.6 GB of free space is needed on Drive.

**Output.**
- `reteco/index/<model>/<domain>.npy`: document vectors (computed once)
- `reteco/runs/dense_<model>_<query>_<split>/<domain>/run.trec`: runs

**Next.** Copy `reteco/runs/` from Drive to `data/drive/reteco/runs/` of the repo.

**New query variants** (`query.ipynb` writes them to `reteco/queries`): "Run all"
again. The corpus vectors are already there, so only the queries are encoded (minutes).

Model: a reasoning embedder built on Qwen3-Embedding-4B. On RECOR the benchmark authors report 0.545 vs. 0.446 for BM25 (with history).

**To keep the session from disconnecting when idle**: after "Run all", open the browser console
(Cmd+Option+J → Console), paste the code below and press Enter. Once a minute it clicks the
resources button while the runtime is connected, and it stops by itself once the notebook has finished
and released the GPU (it does not reconnect, so no compute units are spent). The computer must not
go to sleep, and the tab must stay open.
A log of all output with timestamps is written to `logs/` in the project folder on Drive (`DRIVE`).

```js
// Clicker for Colab: keeps the session from disconnecting when idle while the runtime is connected.
// Paste it into the console of the Colab tab (Cmd+Option+J → Console tab → paste → Enter)
// after the notebook has been started. Once a minute it clicks the resources button in the header.
// It stops by itself once the runtime has disconnected (the notebook finished or crashed and
// released the GPU) and does NOT reconnect, so no compute units are spent on an idle machine.
(() => {
  if (window.__retecoKeepAlive) { console.log("clicker is already running"); return; }
  const btn = () => document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect");
  const disconnected = () => {
    const b = btn();
    if (!b) return true;
    // the button also shows the GPU type ("Reconnect T4"), so only the first word is compared;
    // the non-English labels below are the same button texts in the Russian-locale Colab UI
    const text = (b.textContent || "").trim().split(/\s+/)[0].toLowerCase();
    const tip = (b.getAttribute("tooltiptext") || "").toLowerCase();
    return ["connect", "reconnect", "подключиться", "переподключиться"].includes(text)
      || tip.startsWith("connect to a new") || tip.startsWith("подключиться к");
  };
  let seen = false, clicks = 0;
  window.__retecoKeepAlive = setInterval(() => {
    if (!disconnected()) {
      seen = true;
      btn().click();                       // opens/closes the resources panel; harmless
      clicks += 1;
      console.log(`keep-alive ${new Date().toLocaleTimeString()} · ${clicks}`);
    } else if (seen) {
      clearInterval(window.__retecoKeepAlive);
      window.__retecoKeepAlive = null;
      console.log("runtime disconnected: clicker stopped");
    }
  }, 60 * 1000);
  console.log("clicker started: once a minute; stops by itself after the runtime disconnects");
})();
```

In [ ]:
# ── 1. Settings ──────────────────────────────────────────────────────────────
MODEL = "AQ-MedAI/Diver-Retriever-4B"
BATCH = 32            # "CUDA out of memory" → 16, then 8
MAX_DOC_LEN = 512     # tokens per document; the median document is 20 words
MAX_Q_LEN = 1024      # a query with history can reach ~600 tokens
SPLITS = ["dev", "train"]
DRIVE = "/content/drive/MyDrive/RETECO 2026/reteco"   # project folder on Drive (configurable)

In [ ]:
# ── 1b. Compute-unit safeguard ──────────────────────────────────────────────
# At the end of the notebook the runtime disconnects itself, so an idle GPU does not burn compute units.
# If a cell fails and nothing is run for {IDLE_STOP_MIN} min, it disconnects as well.
# To turn this off, set AUTO_STOP = False in the settings.
NB = "diver"

import threading, time
from IPython import get_ipython
AUTO_STOP = globals().get("AUTO_STOP", True)
IDLE_STOP_MIN = globals().get("IDLE_STOP_MIN", 15)

def stop_runtime(reason):
    print(f"\n[{time.strftime('%H:%M:%S')}] {reason}: disconnecting the runtime to save compute units", flush=True)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print("could not disconnect:", e)

_WD = {"t": None}
def _pre_cell(*a, **k):
    if _WD["t"]:
        _WD["t"].cancel(); _WD["t"] = None
def _post_cell(result):
    if AUTO_STOP and getattr(result, "error_in_exec", None) is not None:
        _WD["t"] = threading.Timer(IDLE_STOP_MIN * 60, stop_runtime,
                                   args=(f"error and {IDLE_STOP_MIN} min without activity",))
        _WD["t"].daemon = True; _WD["t"].start()
_ip = get_ipython()
if _ip is not None and not getattr(_ip, "_reteco_guard", False):
    _ip.events.register("pre_run_cell", _pre_cell)
    _ip.events.register("post_run_cell", _post_cell)
    _ip._reteco_guard = True
print("safeguard:", "the runtime disconnects itself at the end or", IDLE_STOP_MIN, "min after an error"
      if AUTO_STOP else "off (AUTO_STOP = False)")

In [ ]:
# ── 2. Drive, packages, GPU ──────────────────────────────────────────────────
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"   # the HF Xet transport sometimes hangs; plain HTTP is more reliable
from google.colab import drive
drive.mount("/content/drive")
!pip -q install -U sentence-transformers pytrec-eval-terrier
import torch, transformers
from packaging.version import Version
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
DTYPE_KEY = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB,",
      "transformers", transformers.__version__)

In [ ]:
# ── 3. RETECO data, track 2 (≈270 MB, about a minute) ───────────────────────
from huggingface_hub import snapshot_download
snapshot_download("DataScience-UIBK/RETECO-SemEval2027", repo_type="dataset",
                  local_dir="/content/reteco",
                  allow_patterns=["track2_recor/*/documents.jsonl",
                                  "track2_recor/*/benchmark_*.json",
                                  "track2_recor/*/qrels_*.txt"])

In [ ]:
# ── 4. Shared code. Mirrors scripts/retrieve.py and score.py in the repository ──
import os, re, json, time, glob
import numpy as np, pytrec_eval
from tqdm.auto import tqdm

DATA = "/content/reteco/track2_recor"
DOMAINS = ["drones", "politics", "law", "medicalsciences", "hardware", "economics",
           "psychology", "biology", "sustainable_living", "robotics", "earth_science"]  # smallest to largest
# official BM25 "turn + history", dev, nDCG@10, for comparison
BM25_DEV = {"biology": .6065, "drones": .2954, "earth_science": .6264, "economics": .4739,
            "hardware": .3378, "law": .3437, "medicalsciences": .1730, "politics": .3219,
            "psychology": .5726, "robotics": .5123, "sustainable_living": .5531}
BM25_MACRO = {"dev": 0.4379, "train": 0.4539}
RUNS, QUERIES = f"{DRIVE}/runs", f"{DRIVE}/queries"

# First run: the BM25 runs and LLM query rewrites sit next to the notebooks as parts
# of the archive reteco_bundle.tar.xz.part*; they are unpacked into DRIVE once.
_parts = sorted(glob.glob(f"{os.path.dirname(DRIVE)}/reteco_bundle.tar.xz.part*"))
if _parts and not os.path.exists(f"{DRIVE}/.bundle_ok"):
    import tarfile
    with open("/content/bundle.tar.xz", "wb") as _out:
        for _p in _parts:
            _out.write(open(_p, "rb").read())
    with tarfile.open("/content/bundle.tar.xz", "r:xz") as _tf:
        try:
            _tf.extractall(os.path.dirname(DRIVE), filter="data")
        except TypeError:
            _tf.extractall(os.path.dirname(DRIVE))
    open(f"{DRIVE}/.bundle_ok", "w").write("ok")
    print(f"project data unpacked from {len(_parts)} parts → {DRIVE}")
    try:                 # flush the unpacked files to Drive now, before the next notebooks start
        from google.colab import drive as _gd
        _gd.flush_and_unmount(); _gd.mount("/content/drive")
    except Exception as _e:
        print("flush Drive:", _e)
assert os.path.isdir(f"{RUNS}/bm25_hist_dev"), (
    f"no project data in {DRIVE}: the notebooks and reteco_bundle.tar.xz.part00…03 must be "
    "in the parent folder of DRIVE (default: MyDrive/RETECO 2026) of the account mounted by drive.mount")
os.makedirs(RUNS, exist_ok=True); os.makedirs(QUERIES, exist_ok=True)

def load_corpus(dom):
    ids, texts = [], []
    for ln in open(f"{DATA}/{dom}/documents.jsonl", encoding="utf-8"):
        if ln.strip():
            d = json.loads(ln); ids.append(d["doc_id"]); texts.append(d["content"])
    return ids, texts

def topics(dom, split, qset):
    """Turns without gold documents are skipped; history is appended as in the organisers' baseline.
    qset: "query", "hist" or a folder name in Drive/reteco/queries (variants from rewrite.py)."""
    given = None
    if qset not in ("query", "hist"):
        with open(f"{QUERIES}/{qset}/{split}.jsonl", encoding="utf-8") as f:
            # key (domain, topic_id): in the BRIGHT domains topic_ids repeat across domains
            given = {(r["domain"], r["topic_id"]): r["text"] for r in map(json.loads, filter(str.strip, f))}
    ids, texts = [], []
    for c in json.load(open(f"{DATA}/{dom}/benchmark_{split}.json", encoding="utf-8")):
        for t in c["turns"]:
            if not (t.get("gold_doc_ids") or t.get("supporting_doc_ids")):
                continue
            qid = f"{c['id']}_turn_{t['turn_id']}"
            if given is not None:
                text = given[(dom, qid)]
            else:
                text, h = t["query"], t.get("conversation_history", "")
                if qset == "hist" and h and h != "No previous conversation.":
                    text += f"\n\nConversation History:\n{h}"
            ids.append(qid); texts.append(text)
    return ids, texts

def ndcg10(run, dom, split):
    """run: {qid: {doc: score}}. A turn missing from the run scores 0, as on the leaderboard."""
    qrels = {}
    for ln in open(f"{DATA}/{dom}/qrels_{split}.txt"):
        p = ln.split()
        if len(p) == 4:
            qrels.setdefault(p[0], {})[p[2]] = int(p[3])
    res = pytrec_eval.RelevanceEvaluator(qrels, {"ndcg_cut.10"}).evaluate(run)
    return sum(v["ndcg_cut_10"] for v in res.values()) / len(qrels)

def read_run(path):
    run = {}
    for ln in open(path, encoding="utf-8"):
        p = ln.split()
        if len(p) == 6:
            run.setdefault(p[0], {})[p[2]] = float(p[4])
    return run

def write_run(path, run, tag):
    """Writes to a temporary file first, then renames it, so an interrupted write never
    leaves a partial file. Scores are written strictly decreasing and with spare digits:
    pytrec_eval breaks score ties by docno in reverse order, which in five domains would
    systematically rank _neg_ above _doc_. Tied scores keep the dict order (that is,
    the first-stage order)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        for q, docs in run.items():
            last = None
            for r, (d, sc) in enumerate(sorted(docs.items(), key=lambda x: -x[1]), 1):
                sc = float(sc)
                if last is not None and sc >= last:
                    sc = last - 1e-9 * max(1.0, abs(last))
                last = sc
                f.write(f"{q}\tQ0\t{d}\t{r}\t{sc:.12g}\t{tag}\n")
    os.replace(path + ".tmp", path)

def strata(rs):
    """Slices for the generation tables. All rows ("total"); for 2c, by the qrels label (all/part/none)
    plus "clean" misses (none, and the gold never appeared in earlier turns, so the answer cannot be
    recovered from the history); for 2b and nodoc, the first turn (no history) and the rest."""
    yield "total", rs
    if rs and rs[0]["mode"] == "2c":
        for x in ("all", "part", "none"):
            yield x, [r for r in rs if r["suff"] == x]
        yield "clean", [r for r in rs if r["suff"] == "none" and r.get("gold_in_prior") == 0]
    else:
        yield "T1", [r for r in rs if r.get("turn_id") == 1]
        yield "T2+", [r for r in rs if (r.get("turn_id") or 0) > 1]

# ── log: everything the cells print, timestamped → Drive/reteco/logs ────────
import sys
class _Tee:
    def __init__(self, out, path):
        self.out, self.f, self.bol = out, open(path, "a", encoding="utf-8"), True
    def write(self, x):
        self.out.write(x)
        try:
            for part in x.splitlines(True):
                if self.bol:
                    self.f.write(time.strftime("%m-%d %H:%M:%S "))
                self.f.write(part)
                self.bol = part.endswith("\n")
            self.f.flush()
        except Exception:
            pass
        return len(x)
    def flush(self):
        self.out.flush()
    def __getattr__(self, k):
        return getattr(self.out, k)
os.makedirs(f"{DRIVE}/logs", exist_ok=True)
if not isinstance(getattr(sys.stdout, "_reteco_log", None), str):
    LOG_PATH = f"{DRIVE}/logs/{globals().get('NB', 'notebook')}_{time.strftime('%Y%m%d_%H%M%S')}.log"
    sys.stdout = _Tee(sys.stdout, LOG_PATH)
    sys.stdout._reteco_log = LOG_PATH
LOG_PATH = sys.stdout._reteco_log
print("log:", LOG_PATH)

In [ ]:
# ── 5. Model. Query and document prompts come from the model's own config ─────
from sentence_transformers import SentenceTransformer
if "model" not in globals():      # re-running "Run all" does not load a second copy onto the GPU
    model = SentenceTransformer(MODEL, device="cuda", model_kwargs={DTYPE_KEY: torch.float16},
                                trust_remote_code=True)
assert next(model.parameters()).dtype == torch.float16, "the model was not loaded in fp16"
Q_PROMPT = (model.prompts or {}).get("query", "")
D_PROMPT = (model.prompts or {}).get("document", "")
print("query prompt:   ", repr(Q_PROMPT))
print("document prompt:", repr(D_PROMPT))
SLUG = re.sub(r"[^A-Za-z0-9._-]", "_", MODEL)
IDX = f"{DRIVE}/index/{SLUG}"
os.makedirs(IDX, exist_ok=True)

def encode(texts, prompt, max_len):
    model.max_seq_length = max_len
    e = model.encode(texts, prompt=prompt, batch_size=BATCH, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=True)
    assert not np.isnan(e).any(), "NaN in vectors: fp16 overflowed; use an A100 and bfloat16"
    return e

def search(emb, ids, qids, qtexts, topk=100):
    # scores in fp32: in fp16 close cosines collide, and pytrec_eval breaks ties by docno
    Q = torch.from_numpy(encode(qtexts, Q_PROMPT, MAX_Q_LEN)).cuda().float()
    E = torch.from_numpy(emb).cuda().float()
    run = {}
    for i in range(0, len(qids), 64):
        v, ix = (Q[i:i + 64] @ E.T).topk(min(topk, E.shape[0]), dim=1)
        for j, q in enumerate(qids[i:i + 64]):
            run[q] = {ids[k]: float(s) for k, s in zip(ix[j].tolist(), v[j].tolist())}
    del E, Q; torch.cuda.empty_cache()
    return run

def run_dir(qset, split):
    return f"{RUNS}/dense_{SLUG}_{qset}_{split}"

In [ ]:
# ── 6. Corpus vectors → Drive. A dev check after each domain ───────────────
sizes = {d: sum(1 for _ in open(f"{DATA}/{d}/documents.jsonl")) for d in DOMAINS}
rate = None
for dom in DOMAINS:
    out = f"{IDX}/{dom}.npy"
    if os.path.exists(out):
        print(f"[done] {dom}"); continue
    ids, docs = load_corpus(dom)
    t = time.time()
    parts = []
    for b in range(0, len(docs), 20000):     # in blocks: an interruption loses at most one block
        pp = f"{IDX}/{dom}.part{b // 20000:03d}.npy"
        if not os.path.exists(pp):
            with open(pp + ".tmp", "wb") as f:
                np.save(f, encode(docs[b:b + 20000], D_PROMPT, MAX_DOC_LEN).astype(np.float16))
            os.replace(pp + ".tmp", pp)
            print(f"  {dom}: {min(b + 20000, len(docs))}/{len(docs)} docs")
        parts.append(pp)
    emb = np.concatenate([np.load(p) for p in parts])
    sec = time.time() - t
    json.dump(ids, open(f"{IDX}/{dom}.ids.json", "w"))
    with open(out + ".tmp", "wb") as f:
        np.save(f, emb)
    os.replace(out + ".tmp", out)
    for p in parts:
        os.remove(p)
    rate = len(docs) / sec
    left = sum(sizes[d] for d in DOMAINS if not os.path.exists(f"{IDX}/{d}.npy"))
    # check: dense "turn + history" retrieval on dev against the official BM25
    qids, qtexts = topics(dom, "dev", "hist")
    run = search(emb, ids, qids, qtexts)
    write_run(f"{run_dir('hist', 'dev')}/{dom}/run.trec", run, "dense_hist")
    print(f"{dom}: {len(docs)} docs in {sec/60:.1f} min ({rate:.0f} docs/s), ≈{left/rate/60:.0f} min left · "
          f"nDCG@10 dev {ndcg10(run, dom, 'dev'):.4f} vs BM25 {BM25_DEV[dom]:.4f}")
print("corpus vectors ready:", IDX)

In [ ]:
# ── 7. Runs: all query sets × splits → Drive/reteco/runs ────────────────────
# Domain is the outer loop: each domain's vectors are read from Drive once for all query sets.
extra = sorted(os.path.basename(p) for p in glob.glob(f"{QUERIES}/*")
               if os.path.isdir(p) and os.path.basename(p) != "raw")
JOBS = [(q, s) for q in ["hist", "query"] + extra for s in SPLITS
        if q in ("query", "hist") or os.path.exists(f"{QUERIES}/{q}/{s}.jsonl")]
print("query sets × splits:", JOBS)
for dom in DOMAINS:
    todo = [(q, s) for q, s in JOBS if not os.path.exists(f"{run_dir(q, s)}/{dom}/run.trec")]
    if not todo:
        continue
    ids, emb = json.load(open(f"{IDX}/{dom}.ids.json")), np.load(f"{IDX}/{dom}.npy")
    for qset, split in todo:
        qids, qtexts = topics(dom, split, qset)
        write_run(f"{run_dir(qset, split)}/{dom}/run.trec", search(emb, ids, qids, qtexts), f"dense_{qset}")
    print(f"{dom}: {len(todo)} runs")
    del emb

summary = []
for qset, split in JOBS:
    rd = run_dir(qset, split)
    macro = sum(ndcg10(read_run(f"{rd}/{d}/run.trec"), d, split) for d in DOMAINS) / len(DOMAINS)
    json.dump({"model": MODEL, "query_prompt": Q_PROMPT, "doc_prompt": D_PROMPT,
               "max_doc_len": MAX_DOC_LEN, "max_q_len": MAX_Q_LEN, "dtype": "float16",
               "query_set": qset, "split": split, "topk": 100,
               "gpu": torch.cuda.get_device_name(0),
               "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
               "date": time.strftime("%Y-%m-%dT%H:%M:%S")},
              open(f"{rd}/config.json", "w"), ensure_ascii=False, indent=2)
    summary.append((os.path.basename(rd), macro, BM25_MACRO[split]))

In [ ]:
# ── 8. Summary ──────────────────────────────────────────────────────────────
print(f"{'run':<60}{'nDCG@10':>9}{'BM25':>8}{'delta':>9}")
for name, m, b in summary:
    print(f"{name:<60}{m:>9.4f}{b:>8.4f}{m - b:>+9.4f}")
print("\nNext: copy", RUNS, "from Drive to data/drive/reteco/runs/ of the repo.")

In [ ]:
# ── End. Everything is on Drive; the runtime disconnects so that no compute units are spent ──
print("done:", time.strftime("%Y-%m-%d %H:%M:%S"), "· log:", globals().get("LOG_PATH", "—"))
if AUTO_STOP:
    stop_runtime("notebook finished")